In [128]:
# from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import ticker

path_건축물대장 = "../data/processed/시군구_주거여부_연도별_층별면적_집계.parquet"
path_주민등록인구_raw = (
    "../data/raw/행정구역(시군구)별1세별 주민등록인구/101_DT_1B04006_Y_2024.csv"
)
path_한국행정구역분류 = "../data/processed/code_kcad_sgg_2024.csv"

In [129]:
# Open a DuckDB in-memory connection
con = duckdb.connect()

rel_building = con.read_parquet(path_건축물대장)
df_pop = pd.read_csv(path_주민등록인구_raw, skiprows=2, encoding="euc-kr")
rel_kcad = con.read_csv(path_한국행정구역분류, dtype=["string"] * 12)

In [130]:
df_pop.head()

,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명)
0,'00,전국,'000,계,2024,51217221.0,25498324.0,25718897.0
1,'00,전국,'0401,0세,2024,235337.0,120438.0,114899.0
2,'00,전국,'0402,1세,2024,234405.0,120019.0,114386.0
3,'00,전국,'0403,2세,2024,254654.0,130169.0,124485.0
4,'00,전국,'0404,3세,2024,267456.0,137079.0,130377.0


In [131]:
df_pop["총인구수 (명)"] = df_pop["총인구수 (명)"].astype(int)
df_pop["남자인구수 (명)"] = df_pop["남자인구수 (명)"].astype(int)
df_pop["여자인구수 (명)"] = df_pop["여자인구수 (명)"].astype(int)
df_pop.head()


,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명)
0,'00,전국,'000,계,2024,51217221,25498324,25718897
1,'00,전국,'0401,0세,2024,235337,120438,114899
2,'00,전국,'0402,1세,2024,234405,120019,114386
3,'00,전국,'0403,2세,2024,254654,130169,124485
4,'00,전국,'0404,3세,2024,267456,137079,130377


In [132]:
df_pop["시군구_코드"] = df_pop["C행정구역(시군구)별"].str.strip("'")
df_pop["시군구_코드"].unique()

array(['00', '11', '11110', '11140', '11170', '11200', '11215', '11230',
       '11260', '11290', '11305', '11320', '11350', '11380', '11410',
       '11440', '11470', '11500', '11530', '11545', '11560', '11590',
       '11620', '11650', '11680', '11710', '11740', '26', '26110',
       '26140', '26170', '26200', '26230', '26260', '26290', '26320',
       '26350', '26380', '26410', '26440', '26470', '26500', '26530',
       '26710', '27', '27110', '27140', '27170', '27200', '27230',
       '27260', '27290', '27710', '27720', '28', '28110', '28114',
       '28118', '28140', '28177', '28185', '28200', '28237', '28245',
       '28260', '28265', '28710', '28720', '29', '29110', '29140',
       '29155', '29170', '29200', '30', '30110', '30140', '30170',
       '30200', '30230', '31', '31110', '31140', '31170', '31200',
       '31710', '36', '36110', '41', '41105', '41110', '41111', '41113',
       '41115', '41117', '41130', '41131', '41133', '41135', '41150',
       '41170', '41171', '41173'

In [133]:
region_5digit = [x for x in df_pop["시군구_코드"].unique().tolist() if (len(x) == 5)]

df_pop[df_pop["시군구_코드"].isin(region_5digit) & df_pop["C연령별"].eq("'000")][
    "총인구수 (명)"
].sum()

np.int64(61067755)

시군구 인구는 비자치구 인구가 중복되어 있어 (수원시 구, 창원시 구 등) 건축물대장과 비교 가능한 형태로 일치시킬 필요


In [134]:
df = rel_kcad.df()
df["행정구역분류_4자리"] = df["행정구역분류"].str.slice(0, 4)
print(df.shape)
df0 = df[df["행정구역분류"].str.endswith("0")]
dfx = df[~df["행정구역분류"].str.endswith("0")]
print(df0.shape)
display(df0.head())
print(dfx.shape)
display(dfx.head())

(264, 13)
(229, 13)


,시군구코드,시도,시군구,행정구역명,행정동(행정기관명),법정동,행정구역분류,행정기관코드,행정기관 생성일,법정동코드,법정동 관할구역분할여부,행정동 영문명칭,행정구역분류_4자리
0,11110,서울특별시,종로구,종로구,종로구,종로구,11010,1111000000,19880423,1111000000,None,Jongno-gu,1101
1,11140,서울특별시,중구,중구,중구,중구,11020,1114000000,19880423,1114000000,None,Jung-gu,1102
2,11170,서울특별시,용산구,용산구,용산구,용산구,11030,1117000000,19880423,1117000000,None,Yongsan-gu,1103
3,11200,서울특별시,성동구,성동구,성동구,성동구,11040,1120000000,19880423,1120000000,None,Seongdong-gu,1104
4,11215,서울특별시,광진구,광진구,광진구,광진구,11050,1121500000,19950301,1121500000,None,Gwangjin-gu,1105


(35, 13)


,시군구코드,시도,시군구,행정구역명,행정동(행정기관명),법정동,행정구역분류,행정기관코드,행정기관 생성일,법정동코드,법정동 관할구역분할여부,행정동 영문명칭,행정구역분류_4자리
77,41111,경기도,수원시 장안구,수원시 장안구,수원시 장안구,수원시 장안구,31011,4111100000,19880701,4111100000,None,Jangan-gu,3101
78,41113,경기도,수원시 권선구,수원시 권선구,수원시 권선구,수원시 권선구,31012,4111300000,19880701,4111300000,None,Gwonseon-gu,3101
79,41115,경기도,수원시 팔달구,수원시 팔달구,수원시 팔달구,수원시 팔달구,31013,4111500000,19930115,4111500000,None,Paldal-gu,3101
80,41117,경기도,수원시 영통구,수원시 영통구,수원시 영통구,수원시 영통구,31014,4111700000,20031124,4111700000,None,Yeongtong-gu,3101
82,41131,경기도,성남시 수정구,성남시 수정구,성남시 수정구,성남시 수정구,31021,4113100000,19890501,4113100000,None,Sujeong-gu,3102


행정구역분류 코드 마지막 자리가 0이면 시군구(자치구), 0이 아니면 자치구가 아닌 구(비자치구)


In [135]:
df_pop[df_pop["시군구_코드"].isin(df0.시군구코드) & df_pop["C연령별"].eq("'000")][
    "총인구수 (명)"
].sum()

np.int64(51217221)

건축물대장은 비자치구 시군구코드를 사용하고 있으므로, 비자치구가 있는 경우 상위 시군구 코드를 제외하여 중복 없이 인구와 비교


In [136]:
df0_without_subregion = df0[~df0["행정구역분류_4자리"].isin(dfx["행정구역분류_4자리"])]
df0_without_subregion.shape

(217, 13)

In [195]:
df_kcad_sgg = pd.concat([df0_without_subregion, dfx], axis="index").sort_values(
    by="행정구역분류"
)
df_kcad_sgg.shape

(252, 13)

In [196]:
df_pop[
    df_pop["시군구_코드"].isin(df_kcad_sgg.시군구코드) & df_pop["C연령별"].eq("'000")
]["총인구수 (명)"].sum()

np.int64(51217221)

In [197]:
df_kcad_sgg

,시군구코드,시도,시군구,행정구역명,행정동(행정기관명),법정동,행정구역분류,행정기관코드,행정기관 생성일,법정동코드,법정동 관할구역분할여부,행정동 영문명칭,행정구역분류_4자리
0,11110,서울특별시,종로구,종로구,종로구,종로구,11010,1111000000,19880423,1111000000,None,Jongno-gu,1101
1,11140,서울특별시,중구,중구,중구,중구,11020,1114000000,19880423,1114000000,None,Jung-gu,1102
2,11170,서울특별시,용산구,용산구,용산구,용산구,11030,1117000000,19880423,1117000000,None,Yongsan-gu,1103
3,11200,서울특별시,성동구,성동구,성동구,성동구,11040,1120000000,19880423,1120000000,None,Seongdong-gu,1104
4,11215,서울특별시,광진구,광진구,광진구,광진구,11050,1121500000,19950301,1121500000,None,Gwangjin-gu,1105
...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,48870,경상남도,함양군,함양군,함양군,함양군,38580,4887000000,19880423,4887000000,None,Hamyang-gun,3858
260,48880,경상남도,거창군,거창군,거창군,거창군,38590,4888000000,19880423,4888000000,None,Geochang-gun,3859
261,48890,경상남도,합천군,합천군,합천군,합천군,38600,4889000000,19880423,4889000000,None,Hapcheon-gun,3860
262,50110,제주특별자치도,제주시,제주시,제주시,제주시,39010,5011000000,20060701,5011000000,None,Jeju-si,3901


In [140]:
region_country = ["00"]

df_pop[df_pop["시군구_코드"].isin(region_country) & df_pop["C연령별"].eq("'000")][
    "총인구수 (명)"
].sum()

np.int64(51217221)

In [141]:
region_sido = [
    x
    for x in df_pop["시군구_코드"].unique().tolist()
    if (len(x) == 2) & (x not in region_country)
]

df_pop[df_pop["시군구_코드"].isin(region_sido) & df_pop["C연령별"].eq("'000")][
    "총인구수 (명)"
].sum()

np.int64(51217221)

In [198]:
# with 연령별

df_pop[df_pop["시군구_코드"].isin(region_sido) & df_pop["연령별"].str.contains("세")][
    "총인구수 (명)"
].sum()

np.int64(51217221)

In [144]:
df_pop[
    df_pop["시군구_코드"].isin(df_kcad_sgg.시군구코드)
    & df_pop["연령별"].str.contains("세")
]["총인구수 (명)"].sum()

np.int64(51217221)

In [163]:
df_pop_country = df_pop[
    df_pop["시군구_코드"].isin(region_country) & df_pop["연령별"].str.contains("세")
].copy()
df_pop_sido = df_pop[
    df_pop["시군구_코드"].isin(region_sido) & df_pop["연령별"].str.contains("세")
].copy()
df_pop_sgg = df_pop[
    df_pop["시군구_코드"].isin(df_kcad_sgg.시군구코드)
    & df_pop["연령별"].str.contains("세")
].copy()

df_pop_country["연령_int"] = df_pop_country["연령별"].str.strip("세 이상").astype(int)
df_pop_sido["연령_int"] = df_pop_sido["연령별"].str.strip("세 이상").astype(int)
df_pop_sgg["연령_int"] = df_pop_sgg["연령별"].str.strip("세 이상").astype(int)

print(df_pop_country["총인구수 (명)"].sum())
print(df_pop_sido["총인구수 (명)"].sum())
print(df_pop_sgg["총인구수 (명)"].sum())

51217221
51217221
51217221


In [164]:
df_pop_country

,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),시군구_코드,연령_int
1,'00,전국,'0401,0세,2024,235337,120438,114899,00,0
2,'00,전국,'0402,1세,2024,234405,120019,114386,00,1
3,'00,전국,'0403,2세,2024,254654,130169,124485,00,2
4,'00,전국,'0404,3세,2024,267456,137079,130377,00,3
5,'00,전국,'0405,4세,2024,279924,143167,136757,00,4
...,...,...,...,...,...,...,...,...,...,...
97,'00,전국,'4302,96세,2024,15836,2871,12965,00,96
98,'00,전국,'4303,97세,2024,10996,1871,9125,00,97
99,'00,전국,'4304,98세,2024,6369,1020,5349,00,98
100,'00,전국,'4305,99세,2024,4224,631,3593,00,99


In [165]:
df_pop_sido

,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),시군구_코드,연령_int
103,'11,서울특별시,'0401,0세,2024,39588,20193,19395,11,0
104,'11,서울특별시,'0402,1세,2024,37696,19323,18373,11,1
105,'11,서울특별시,'0403,2세,2024,40295,20593,19702,11,2
106,'11,서울특별시,'0404,3세,2024,42163,21640,20523,11,3
107,'11,서울특별시,'0405,4세,2024,43155,22167,20988,11,4
...,...,...,...,...,...,...,...,...,...,...
29881,'50,제주특별자치도,'4302,96세,2024,223,27,196,50,96
29882,'50,제주특별자치도,'4303,97세,2024,173,15,158,50,97
29883,'50,제주특별자치도,'4304,98세,2024,103,9,94,50,98
29884,'50,제주특별자치도,'4305,99세,2024,67,5,62,50,99


In [166]:
rel_building.show()

┌─────────────┬───────────┬───────────────┬─────────┬───────────────┐
│ 시군구_코드 │ 주거_여부 │ 사용승인_연도 │ 층_빈도 │    총면적     │
│   varchar   │  varchar  │    varchar    │  int64  │ decimal(38,3) │
├─────────────┼───────────┼───────────────┼─────────┼───────────────┤
│ 11110       │ 비주거    │ 1899          │       3 │       773.560 │
│ 11110       │ 비주거    │ 1911          │       5 │       201.810 │
│ 11110       │ 비주거    │ 1912          │       8 │      2818.970 │
│ 11110       │ 비주거    │ 1914          │       1 │        17.550 │
│ 11110       │ 비주거    │ 1916          │       1 │        16.600 │
│ 11110       │ 비주거    │ 1917          │       7 │       259.980 │
│ 11110       │ 비주거    │ 1918          │       2 │       122.310 │
│ 11110       │ 비주거    │ 1922          │       3 │      1263.470 │
│ 11110       │ 비주거    │ 1923          │       4 │       188.200 │
│ 11110       │ 비주거    │ 1925          │       1 │        41.720 │
│   ·         │  ·        │  ·            │       · │           ·   │
│

In [167]:
query = f"""
SELECT
    *,
    2024 - CAST(사용승인_연도 AS INT) AS age,
    CASE
        WHEN 2024 - CAST(사용승인_연도 AS INT) BETWEEN 0 AND 99 THEN
            2024 - CAST(사용승인_연도 AS INT)
        WHEN 2024 - CAST(사용승인_연도 AS INT) >= 100 THEN
            100
        ELSE NULL
    END AS 연령_int
FROM rel_building
WHERE 시군구_코드 IN {tuple(df_sgg.시군구코드)}
"""
rel1 = con.sql(query)
rel1.show()

┌─────────────┬───────────┬───────────────┬─────────┬───────────────┬───────┬──────────┐
│ 시군구_코드 │ 주거_여부 │ 사용승인_연도 │ 층_빈도 │    총면적     │  age  │ 연령_int │
│   varchar   │  varchar  │    varchar    │  int64  │ decimal(38,3) │ int32 │  int32   │
├─────────────┼───────────┼───────────────┼─────────┼───────────────┼───────┼──────────┤
│ 11110       │ 비주거    │ 1899          │       3 │       773.560 │   125 │      100 │
│ 11110       │ 비주거    │ 1911          │       5 │       201.810 │   113 │      100 │
│ 11110       │ 비주거    │ 1912          │       8 │      2818.970 │   112 │      100 │
│ 11110       │ 비주거    │ 1914          │       1 │        17.550 │   110 │      100 │
│ 11110       │ 비주거    │ 1916          │       1 │        16.600 │   108 │      100 │
│ 11110       │ 비주거    │ 1917          │       7 │       259.980 │   107 │      100 │
│ 11110       │ 비주거    │ 1918          │       2 │       122.310 │   106 │      100 │
│ 11110       │ 비주거    │ 1922          │       3 │      1263.470 

In [168]:
query = """
SELECT
    시군구_코드,
    연령_int,
    SUM(CASE WHEN 주거_여부 = '주거'   THEN 총면적 ELSE 0 END) AS 주거_총면적,
    SUM(CASE WHEN 주거_여부 = '비주거' THEN 총면적 ELSE 0 END) AS 비주거_총면적,
FROM rel1
GROUP BY ALL
ORDER BY ALL;
"""
rel_sgg = con.sql(query)
rel_sgg.show()

┌─────────────┬──────────┬───────────────┬───────────────┐
│ 시군구_코드 │ 연령_int │  주거_총면적  │ 비주거_총면적 │
│   varchar   │  int32   │ decimal(38,3) │ decimal(38,3) │
├─────────────┼──────────┼───────────────┼───────────────┤
│ 11110       │        0 │     23553.755 │     27683.515 │
│ 11110       │        1 │     28244.830 │     78295.285 │
│ 11110       │        2 │     48558.820 │    142101.414 │
│ 11110       │        3 │     39329.326 │     72709.715 │
│ 11110       │        4 │     41803.624 │    208920.746 │
│ 11110       │        5 │     64637.196 │    171743.841 │
│ 11110       │        6 │     25516.150 │    244954.298 │
│ 11110       │        7 │    279777.167 │     87091.336 │
│ 11110       │        8 │     27007.750 │     86170.026 │
│ 11110       │        9 │     57356.062 │    379718.535 │
│   ·         │        · │         ·     │         ·     │
│   ·         │        · │         ·     │         ·     │
│   ·         │        · │         ·     │         ·     │
│ 41630       │

In [248]:
rel_sgg.shape

(23470, 4)

In [250]:
rel_sgg.describe()

┌─────────┬─────────────┬────────────────────┬────────────────────┬────────────────────┐
│  aggr   │ 시군구_코드 │      연령_int      │    주거_총면적     │   비주거_총면적    │
│ varchar │   varchar   │       double       │       double       │       double       │
├─────────┼─────────────┼────────────────────┼────────────────────┼────────────────────┤
│ count   │ 23470       │            23470.0 │            23470.0 │            23470.0 │
│ mean    │ NULL        │ 47.135705155517684 │  83243.03724904133 │  79183.67522560716 │
│ stddev  │ NULL        │ 28.374038513647964 │ 187057.35913951893 │ 149623.75824312377 │
│ min     │ 11110       │                0.0 │                0.0 │                0.0 │
│ max     │ 52800       │              100.0 │         3883444.26 │        2891595.017 │
│ median  │ NULL        │               46.0 │          12859.585 │          15645.966 │
└─────────┴─────────────┴────────────────────┴────────────────────┴────────────────────┘

In [169]:
query = """
SELECT
    LEFT(시군구_코드,2) AS 시군구_코드,
    연령_int,
    SUM(CASE WHEN 주거_여부 = '주거'   THEN 총면적 ELSE 0 END) AS 주거_총면적,
    SUM(CASE WHEN 주거_여부 = '비주거' THEN 총면적 ELSE 0 END) AS 비주거_총면적,
FROM rel1
GROUP BY ALL
ORDER BY ALL;
"""
rel_sido = con.sql(query)
rel_sido.show()

┌─────────────┬──────────┬───────────────┬───────────────┐
│ 시군구_코드 │ 연령_int │  주거_총면적  │ 비주거_총면적 │
│   varchar   │  int32   │ decimal(38,3) │ decimal(38,3) │
├─────────────┼──────────┼───────────────┼───────────────┤
│ 11          │        0 │   1655988.404 │   4191098.949 │
│ 11          │        1 │   3061283.819 │   4360141.522 │
│ 11          │        2 │   4036362.309 │   3418358.659 │
│ 11          │        3 │   5633553.370 │   4484100.019 │
│ 11          │        4 │   5600720.364 │   5873089.668 │
│ 11          │        5 │   6298386.458 │   5023515.516 │
│ 11          │        6 │   5610403.401 │   5540281.734 │
│ 11          │        7 │   5158838.112 │   7313667.071 │
│ 11          │        8 │   6497614.896 │   4738945.786 │
│ 11          │        9 │   5249543.562 │   4351527.989 │
│ ·           │        · │         ·     │         ·     │
│ ·           │        · │         ·     │         ·     │
│ ·           │        · │         ·     │         ·     │
│ 52          │

In [170]:
query = """
SELECT
    연령_int,
    SUM(CASE WHEN 주거_여부 = '주거'   THEN 총면적 ELSE 0 END) AS 주거_총면적,
    SUM(CASE WHEN 주거_여부 = '비주거' THEN 총면적 ELSE 0 END) AS 비주거_총면적,
FROM rel1
GROUP BY ALL
ORDER BY ALL;
"""
rel_country = con.sql(query)
rel_country.show()

┌──────────┬───────────────┬───────────────┐
│ 연령_int │  주거_총면적  │ 비주거_총면적 │
│  int32   │ decimal(38,3) │ decimal(38,3) │
├──────────┼───────────────┼───────────────┤
│        0 │  38241017.143 │  42796012.002 │
│        1 │  41678493.377 │  45646744.849 │
│        2 │  39805526.951 │  50312693.570 │
│        3 │  41078178.241 │  55030977.644 │
│        4 │  43115946.421 │  57885312.317 │
│        5 │  54716751.285 │  62991675.218 │
│        6 │  64374511.863 │  60616460.475 │
│        7 │  59380310.835 │  60177831.695 │
│        8 │  52209042.072 │  53712861.742 │
│        9 │  49685973.237 │  50270026.308 │
│        · │         ·     │         ·     │
│        · │         ·     │         ·     │
│        · │         ·     │         ·     │
│       91 │    205320.114 │     59700.049 │
│       92 │    325505.097 │     66165.305 │
│       93 │    255469.822 │     44427.831 │
│       94 │   1248712.178 │    231328.117 │
│       95 │    198455.093 │     38335.225 │
│       96 │    176776.

In [ ]:
region_country
region_sido
region_sgg = df_kcad_sgg.시군구코드.copy()
age_int = pd.DataFrame({"연령_int": range(100, -1, -1)}).assign(key="dummy")

print(type(region_country))
print(type(region_sido))
print(type(region_sgg))
print(age_int)

<class 'list'>
<class 'list'>
<class 'pandas.core.series.Series'>
     연령_int    key
0       100  dummy
1        99  dummy
2        98  dummy
3        97  dummy
4        96  dummy
..      ...    ...
96        4  dummy
97        3  dummy
98        2  dummy
99        1  dummy
100       0  dummy

[101 rows x 2 columns]


In [ ]:
df_country = (
    pd.DataFrame({"시군구_코드": region_country})
    .assign(key="dummy")
    .merge(age_int, on="key")
    .drop("key", axis=1)
    .merge(
        df_pop_country, how="left", on=["시군구_코드", "연령_int"], suffixes=["", "_p"]
    )
    .merge(rel_country.df(), how="left", on=["연령_int"], suffixes=["", "_b"])
)
df_country

,시군구_코드,연령_int,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),주거_총면적,비주거_총면적
0,00,100,'00,전국,'440,100세 이상,2024,8577,1525,7052,4.895507e+06,8.248558e+05
1,00,99,'00,전국,'4305,99세,2024,4224,631,3593,1.148257e+06,1.574935e+05
2,00,98,'00,전국,'4304,98세,2024,6369,1020,5349,3.035959e+05,6.687462e+04
3,00,97,'00,전국,'4303,97세,2024,10996,1871,9125,2.708361e+05,3.716827e+04
4,00,96,'00,전국,'4302,96세,2024,15836,2871,12965,1.767768e+05,2.750460e+04
...,...,...,...,...,...,...,...,...,...,...,...,...
96,00,4,'00,전국,'0405,4세,2024,279924,143167,136757,4.311595e+07,5.788531e+07
97,00,3,'00,전국,'0404,3세,2024,267456,137079,130377,4.107818e+07,5.503098e+07
98,00,2,'00,전국,'0403,2세,2024,254654,130169,124485,3.980553e+07,5.031269e+07
99,00,1,'00,전국,'0402,1세,2024,234405,120019,114386,4.167849e+07,4.564674e+07


In [229]:
df_country = df_pop_country.merge(
    rel_country.df(), how="left", on=["연령_int"], suffixes=["_p", "_b"]
)
df_country

,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),시군구_코드,연령_int,주거_총면적,비주거_총면적
0,'00,전국,'0401,0세,2024,235337,120438,114899,00,0,3.824102e+07,4.279601e+07
1,'00,전국,'0402,1세,2024,234405,120019,114386,00,1,4.167849e+07,4.564674e+07
2,'00,전국,'0403,2세,2024,254654,130169,124485,00,2,3.980553e+07,5.031269e+07
3,'00,전국,'0404,3세,2024,267456,137079,130377,00,3,4.107818e+07,5.503098e+07
4,'00,전국,'0405,4세,2024,279924,143167,136757,00,4,4.311595e+07,5.788531e+07
...,...,...,...,...,...,...,...,...,...,...,...,...
96,'00,전국,'4302,96세,2024,15836,2871,12965,00,96,1.767768e+05,2.750460e+04
97,'00,전국,'4303,97세,2024,10996,1871,9125,00,97,2.708361e+05,3.716827e+04
98,'00,전국,'4304,98세,2024,6369,1020,5349,00,98,3.035959e+05,6.687462e+04
99,'00,전국,'4305,99세,2024,4224,631,3593,00,99,1.148257e+06,1.574935e+05


In [303]:
df_country.shape

(101, 12)

In [304]:
df_country.count()

시군구_코드         101
연령_int         101
C행정구역(시군구)별    101
행정구역(시군구)별     101
C연령별           101
연령별            101
시점             101
총인구수 (명)       101
남자인구수 (명)      101
여자인구수 (명)      101
주거_총면적         101
비주거_총면적        101
dtype: int64

In [305]:
df_country.dtypes

시군구_코드          object
연령_int           int64
C행정구역(시군구)별     object
행정구역(시군구)별      object
C연령별            object
연령별             object
시점               int64
총인구수 (명)         int64
남자인구수 (명)        int64
여자인구수 (명)        int64
주거_총면적         float64
비주거_총면적        float64
dtype: object

In [321]:
with pd.option_context("display.float_format", "{:.3f}".format):
    print(df_country.sum(numeric_only=True))

연령_int            5050.000
시점              204424.000
총인구수 (명)      51217221.000
남자인구수 (명)     25498324.000
여자인구수 (명)     25718897.000
주거_총면적      1953714084.235
비주거_총면적     1858440857.545
dtype: float64


In [322]:
df_sido = (
    pd.DataFrame({"시군구_코드": region_sido})
    .assign(key="dummy")
    .merge(age_int, on="key")
    .drop("key", axis=1)
    .merge(df_pop_sido, how="left", on=["시군구_코드", "연령_int"], suffixes=["", "_p"])
    .merge(
        rel_sido.df(), how="left", on=["시군구_코드", "연령_int"], suffixes=["", "_b"]
    )
)
df_sido

,시군구_코드,연령_int,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),주거_총면적,비주거_총면적
0,11,100,'11,서울특별시,'440,100세 이상,2024,1484,334,1150,14463.210,24332.780
1,11,99,'11,서울특별시,'4305,99세,2024,613,108,505,1008.780,82.260
2,11,98,'11,서울특별시,'4304,98세,2024,1020,205,815,995.940,21606.620
3,11,97,'11,서울특별시,'4303,97세,2024,1813,388,1425,2588.100,378.530
4,11,96,'11,서울특별시,'4302,96세,2024,2501,572,1929,1427.310,960.360
...,...,...,...,...,...,...,...,...,...,...,...,...
1712,50,4,'50,제주특별자치도,'0405,4세,2024,4241,2206,2035,642526.052,1043066.918
1713,50,3,'50,제주특별자치도,'0404,3세,2024,3940,2019,1921,460862.125,689643.707
1714,50,2,'50,제주특별자치도,'0403,2세,2024,3682,1889,1793,483935.946,743350.259
1715,50,1,'50,제주특별자치도,'0402,1세,2024,3263,1639,1624,593406.586,597061.001


In [323]:
df_sido.shape

(1717, 12)

In [324]:
df_sido.count()

시군구_코드         1717
연령_int         1717
C행정구역(시군구)별    1717
행정구역(시군구)별     1717
C연령별           1717
연령별            1717
시점             1717
총인구수 (명)       1717
남자인구수 (명)      1717
여자인구수 (명)      1717
주거_총면적         1717
비주거_총면적        1717
dtype: int64

In [325]:
with pd.option_context("display.float_format", "{:.3f}".format):
    print(df_sido.sum(numeric_only=True))

연령_int           85850.000
시점             3475208.000
총인구수 (명)      51217221.000
남자인구수 (명)     25498324.000
여자인구수 (명)     25718897.000
주거_총면적      1953714084.235
비주거_총면적     1858440857.545
dtype: float64


In [346]:
df_sgg = (
    pd.DataFrame({"시군구_코드": region_sgg})
    .assign(key="dummy")
    .merge(age_int, on="key")
    .drop("key", axis=1)
    .merge(df_pop_sgg, how="left", on=["시군구_코드", "연령_int"], suffixes=["", "_p"])
    .merge(
        rel_sgg.df(),
        how="left",
        on=["시군구_코드", "연령_int"],
        suffixes=["", "_b"],
        indicator=True,
        validate="one_to_one",
    )
)
df_sgg

,시군구_코드,연령_int,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),주거_총면적,비주거_총면적,_merge
0,11110,100,'11110,종로구,'440,100세 이상,2024,41,13,28,419.540,5662.450,both
1,11110,99,'11110,종로구,'4305,99세,2024,12,4,8,0.000,41.720,both
2,11110,98,'11110,종로구,'4304,98세,2024,29,10,19,23.340,4877.590,both
3,11110,97,'11110,종로구,'4303,97세,2024,49,7,42,18.980,157.250,both
4,11110,96,'11110,종로구,'4302,96세,2024,64,16,48,33.710,126.170,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25447,50130,4,'50130,서귀포시,'0405,4세,2024,1051,569,482,210734.020,280610.568,both
25448,50130,3,'50130,서귀포시,'0404,3세,2024,946,482,464,180601.899,170721.181,both
25449,50130,2,'50130,서귀포시,'0403,2세,2024,869,446,423,127509.468,205229.850,both
25450,50130,1,'50130,서귀포시,'0402,1세,2024,717,377,340,181207.452,161949.803,both


In [347]:
df_sgg.shape

(25452, 13)

In [348]:
df_sgg._merge.value_counts()

_merge
both          23470
left_only      1982
right_only        0
Name: count, dtype: int64

In [349]:
df_sgg.count()

시군구_코드         25452
연령_int         25452
C행정구역(시군구)별    25452
행정구역(시군구)별     25452
C연령별           25452
연령별            25452
시점             25452
총인구수 (명)       25452
남자인구수 (명)      25452
여자인구수 (명)      25452
주거_총면적         23470
비주거_총면적        23470
_merge         25452
dtype: int64

In [350]:
df_sgg.groupby("시군구_코드")[["주거_총면적", "비주거_총면적"]].count().describe()

,주거_총면적,비주거_총면적
count,252.000000,252.000000
mean,93.134921,93.134921
std,12.074703,12.074703
min,54.000000,54.000000
25%,88.000000,88.000000
50%,101.000000,101.000000
75%,101.000000,101.000000
max,101.000000,101.000000


지역에 따라 건축물 연령이 1세 간격으로 모두 존재하지 않는 경우가 있음. 최소가 100세 구간 중 54개 연도고, 절반 이상은 매년 존재함.

해당 경우는 0으로 치환할 필요가 있음.


In [ ]:
df_sgg[["주거_총면적", "비주거_총면적"]] = df_sgg[
    ["주거_총면적", "비주거_총면적"]
].fillna(0)
df_sgg.count()

시군구_코드         25452
연령_int         25452
C행정구역(시군구)별    25452
행정구역(시군구)별     25452
C연령별           25452
연령별            25452
시점             25452
총인구수 (명)       25452
남자인구수 (명)      25452
여자인구수 (명)      25452
주거_총면적         25452
비주거_총면적        25452
_merge         25452
dtype: int64

In [352]:
with pd.option_context("display.float_format", "{:.3f}".format):
    print(df_sgg.sum(numeric_only=True))

연령_int         1272600.000
시점            51514848.000
총인구수 (명)      51217221.000
남자인구수 (명)     25498324.000
여자인구수 (명)     25718897.000
주거_총면적      1953714084.235
비주거_총면적     1858440857.545
dtype: float64


In [353]:
df_sgg[(df_sgg.주거_총면적.isna())]

,시군구_코드,연령_int,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),주거_총면적,비주거_총면적,_merge


In [359]:
cols = ["남자인구수 (명)", "여자인구수 (명)", "주거_총면적", "비주거_총면적"]
cols_ratio = ["비율_" + c for c in cols]
cols_negative = ["비율_음수_" + c for c in cols]
cols_formatted = ["출력용_" + c for c in cols]

df_ratio_country = df_country.copy()
totals = df_ratio_country.groupby("시군구_코드")[cols].transform("sum")

df_ratio_country[cols_ratio] = df_ratio_country[cols] / totals
df_ratio_country[cols_negative] = -df_ratio_country[cols_ratio]
df_ratio_country[cols_formatted] = df_ratio_country[cols_ratio].map(
    lambda v: f"{v * 100:.2f}%"
)
df_ratio_country


,시군구_코드,연령_int,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),...,비율_주거_총면적,비율_비주거_총면적,비율_음수_남자인구수 (명),비율_음수_여자인구수 (명),비율_음수_주거_총면적,비율_음수_비주거_총면적,출력용_남자인구수 (명),출력용_여자인구수 (명),출력용_주거_총면적,출력용_비주거_총면적
0,00,100,'00,전국,'440,100세 이상,2024,8577,1525,7052,...,0.002506,0.000444,-0.000060,-0.000274,-0.002506,-0.000444,0.01%,0.03%,0.25%,0.04%
1,00,99,'00,전국,'4305,99세,2024,4224,631,3593,...,0.000588,0.000085,-0.000025,-0.000140,-0.000588,-0.000085,0.00%,0.01%,0.06%,0.01%
2,00,98,'00,전국,'4304,98세,2024,6369,1020,5349,...,0.000155,0.000036,-0.000040,-0.000208,-0.000155,-0.000036,0.00%,0.02%,0.02%,0.00%
3,00,97,'00,전국,'4303,97세,2024,10996,1871,9125,...,0.000139,0.000020,-0.000073,-0.000355,-0.000139,-0.000020,0.01%,0.04%,0.01%,0.00%
4,00,96,'00,전국,'4302,96세,2024,15836,2871,12965,...,0.000090,0.000015,-0.000113,-0.000504,-0.000090,-0.000015,0.01%,0.05%,0.01%,0.00%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,00,4,'00,전국,'0405,4세,2024,279924,143167,136757,...,0.022069,0.031147,-0.005615,-0.005317,-0.022069,-0.031147,0.56%,0.53%,2.21%,3.11%
97,00,3,'00,전국,'0404,3세,2024,267456,137079,130377,...,0.021026,0.029611,-0.005376,-0.005069,-0.021026,-0.029611,0.54%,0.51%,2.10%,2.96%
98,00,2,'00,전국,'0403,2세,2024,254654,130169,124485,...,0.020374,0.027073,-0.005105,-0.004840,-0.020374,-0.027073,0.51%,0.48%,2.04%,2.71%
99,00,1,'00,전국,'0402,1세,2024,234405,120019,114386,...,0.021333,0.024562,-0.004707,-0.004448,-0.021333,-0.024562,0.47%,0.44%,2.13%,2.46%


In [360]:
df_ratio_sido = df_sido.copy()
totals = df_ratio_sido.groupby("시군구_코드")[cols].transform("sum")

df_ratio_sido[cols_ratio] = df_ratio_sido[cols] / totals
df_ratio_sido[cols_negative] = -df_ratio_sido[cols_ratio]
df_ratio_sido[cols_formatted] = df_ratio_sido[cols_ratio].map(
    lambda v: f"{v * 100:.2f}%"
)
df_ratio_sido

,시군구_코드,연령_int,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),...,비율_주거_총면적,비율_비주거_총면적,비율_음수_남자인구수 (명),비율_음수_여자인구수 (명),비율_음수_주거_총면적,비율_음수_비주거_총면적,출력용_남자인구수 (명),출력용_여자인구수 (명),출력용_주거_총면적,출력용_비주거_총면적
0,11,100,'11,서울특별시,'440,100세 이상,2024,1484,334,1150,...,0.000047,1.015463e-04,-0.000074,-0.000238,-0.000047,-1.015463e-04,0.01%,0.02%,0.00%,0.01%
1,11,99,'11,서울특별시,'4305,99세,2024,613,108,505,...,0.000003,3.432898e-07,-0.000024,-0.000105,-0.000003,-3.432898e-07,0.00%,0.01%,0.00%,0.00%
2,11,98,'11,서울특별시,'4304,98세,2024,1020,205,815,...,0.000003,9.016937e-05,-0.000046,-0.000169,-0.000003,-9.016937e-05,0.00%,0.02%,0.00%,0.01%
3,11,97,'11,서울특별시,'4303,97세,2024,1813,388,1425,...,0.000008,1.579692e-06,-0.000086,-0.000295,-0.000008,-1.579692e-06,0.01%,0.03%,0.00%,0.00%
4,11,96,'11,서울특별시,'4302,96세,2024,2501,572,1929,...,0.000005,4.007802e-06,-0.000127,-0.000400,-0.000005,-4.007802e-06,0.01%,0.04%,0.00%,0.00%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1712,50,4,'50,제주특별자치도,'0405,4세,2024,4241,2206,2035,...,0.024868,3.589292e-02,-0.006580,-0.006073,-0.024868,-3.589292e-02,0.66%,0.61%,2.49%,3.59%
1713,50,3,'50,제주특별자치도,'0404,3세,2024,3940,2019,1921,...,0.017837,2.373129e-02,-0.006022,-0.005732,-0.017837,-2.373129e-02,0.60%,0.57%,1.78%,2.37%
1714,50,2,'50,제주특별자치도,'0403,2세,2024,3682,1889,1793,...,0.018730,2.557938e-02,-0.005635,-0.005350,-0.018730,-2.557938e-02,0.56%,0.54%,1.87%,2.56%
1715,50,1,'50,제주특별자치도,'0402,1세,2024,3263,1639,1624,...,0.022967,2.054543e-02,-0.004889,-0.004846,-0.022967,-2.054543e-02,0.49%,0.48%,2.30%,2.05%


In [361]:
df_ratio_sgg = df_sgg.copy()
totals = df_ratio_sgg.groupby("시군구_코드")[cols].transform("sum")

df_ratio_sgg[cols_ratio] = df_ratio_sgg[cols] / totals
df_ratio_sgg[cols_negative] = -df_ratio_sgg[cols_ratio]
df_ratio_sgg[cols_formatted] = df_ratio_sgg[cols_ratio].map(lambda v: f"{v * 100:.2f}%")
df_ratio_sgg

,시군구_코드,연령_int,C행정구역(시군구)별,행정구역(시군구)별,C연령별,연령별,시점,총인구수 (명),남자인구수 (명),여자인구수 (명),...,비율_주거_총면적,비율_비주거_총면적,비율_음수_남자인구수 (명),비율_음수_여자인구수 (명),비율_음수_주거_총면적,비율_음수_비주거_총면적,출력용_남자인구수 (명),출력용_여자인구수 (명),출력용_주거_총면적,출력용_비주거_총면적
0,11110,100,'11110,종로구,'440,100세 이상,2024,41,13,28,...,0.000084,0.000560,-0.000195,-0.000390,-0.000084,-0.000560,0.02%,0.04%,0.01%,0.06%
1,11110,99,'11110,종로구,'4305,99세,2024,12,4,8,...,0.000000,0.000004,-0.000060,-0.000111,-0.000000,-0.000004,0.01%,0.01%,0.00%,0.00%
2,11110,98,'11110,종로구,'4304,98세,2024,29,10,19,...,0.000005,0.000482,-0.000150,-0.000265,-0.000005,-0.000482,0.02%,0.03%,0.00%,0.05%
3,11110,97,'11110,종로구,'4303,97세,2024,49,7,42,...,0.000004,0.000016,-0.000105,-0.000585,-0.000004,-0.000016,0.01%,0.06%,0.00%,0.00%
4,11110,96,'11110,종로구,'4302,96세,2024,64,16,48,...,0.000007,0.000012,-0.000240,-0.000669,-0.000007,-0.000012,0.02%,0.07%,0.00%,0.00%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25447,50130,4,'50130,서귀포시,'0405,4세,2024,1051,569,482,...,0.027963,0.029032,-0.006213,-0.005330,-0.027963,-0.029032,0.62%,0.53%,2.80%,2.90%
25448,50130,3,'50130,서귀포시,'0404,3세,2024,946,482,464,...,0.023965,0.017663,-0.005263,-0.005130,-0.023965,-0.017663,0.53%,0.51%,2.40%,1.77%
25449,50130,2,'50130,서귀포시,'0403,2세,2024,869,446,423,...,0.016920,0.021233,-0.004870,-0.004677,-0.016920,-0.021233,0.49%,0.47%,1.69%,2.12%
25450,50130,1,'50130,서귀포시,'0402,1세,2024,717,377,340,...,0.024045,0.016755,-0.004117,-0.003759,-0.024045,-0.016755,0.41%,0.38%,2.40%,1.68%


In [362]:
df_ratio_country.to_excel("../data/processed/건축물_인구_연령_비율_전국.xlsx")
df_ratio_sido.to_excel("../data/processed/건축물_인구_연령_비율_시도.xlsx")
df_ratio_sgg.to_excel("../data/processed/건축물_인구_연령_비율_시군구.xlsx")